# 33. GraphRAG with Local Attribution

Keep the source chunk on every edge so each fact in the answer can be traced to the original text.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

A graph answer is only trustworthy if you can trace each fact to its source. **Local attribution** keeps the originating chunk id on every edge. When the LLM uses a fact, the citation points to the exact source text, so users (and tests) can verify it.

## How it works

1. During extraction store `source=chunk_id` on every edge
2. Retrieve facts together with their source ids
3. Prompt the LLM with `[chunk_id] fact` lines
4. Require citations in the answer
5. Show the cited chunks for verification

```text
 (hnsw)--[is]-->(graph index)   source=chunk 3
 answer: "HNSW is a layered graph index [3]"  --> show chunk 3 text
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** `pip install networkx`

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 16:05:40.394000 16049 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


In [2]:
import networkx as nx

G = nx.MultiDiGraph()
for i, chunk in enumerate(chunks[:15]):
    for s, r, o in extract_triples(chunk):
        G.add_edge(s, o, relation=r, source=i)          # <-- provenance lives on the edge
print(G.number_of_edges(), "attributed edges")

59 attributed edges


In [3]:
def attributed_facts(query, k=8):
    rel = [(f"{u} {d['relation']} {v}", d["source"]) for u, v, d in G.edges(data=True)]
    store_rel = build_store([t for t, _ in rel], [{"source": s} for _, s in rel])
    return [(h["text"], h["meta"]["source"]) for h in store_rel.search(query, k)]

query = "What does the handbook say about checking answers against sources?"
facts = attributed_facts(query)
numbered = "\n".join(f"[{src}] {fact}" for fact, src in facts)
print(llm(f"Facts (each prefixed with its source chunk id):\n{numbered}\n\nQuestion: {query}\n\nAnswer using the facts and cite chunk ids like [3].", system=RAG_SYSTEM))

I don't know. The provided context doesn't mention anything about checking answers against sources, nor does it provide information on what the handbook says about this topic.


Verify: print the cited chunks so a reader can check the claim against the original text.

In [4]:
for src in sorted({s for _, s in facts}):
    print(f"--- chunk {src}\n{chunks[src]}\n")

--- chunk 1
has four stages: ingest documents, split them into chunks, index the chunks as embeddings, and at query time retrieve the closest chunks and generate an answer from them.

## Embeddings

An embedding model maps text to a dense vector so that texts with similar meaning land close together. The all-MiniLM-L6-v2 model produces 384-dimensional vectors and is popular for local experiments because it is small and fast. Similarity between two vectors is usually measured with cosine similarity, which

--- chunk 9
earches with the embedding of that answer, because an answer looks more like a document than a question does.

## Evaluating RAG Systems

A RAG system has two parts to evaluate: retrieval and generation. Retrieval is measured with hit rate, recall at k and mean reciprocal rank, which averages 1 divided by the rank of the first correct result. Generation is measured with faithfulness, which checks that every claim is supported by the context, and answer relevancy, which che

## Audit: is every cited id real and relevant?

In [5]:
import re
cited = sorted({int(x) for x in re.findall(r"\[(\d+)\]", llm(f"Facts:\n{numbered}\n\nQuestion: {query}\n\nAnswer using the facts and cite chunk ids like [3].", system=RAG_SYSTEM))})
print("cited:", cited, "| valid ids:", all(0 <= c < len(chunks) for c in cited))

cited: [] | valid ids: True


## Provenance report
For each fact, print which chunk it came from and whether the chunk text really contains the subject and object.

In [6]:
import pandas as pd
rows = []
for fact, src in facts:
    parts = fact.split(" ")
    rows.append({"fact": fact[:60], "source": src, "subject_in_source": parts[0] in chunks[src].lower()})
pd.DataFrame(rows)

,fact,source,subject_in_source
0,grounding techniques include requiring citatio...,11,True
1,answer answers question,10,True
2,answer looks like document,9,True
3,agent checks whether it has enough evidence,13,True
4,grounding techniques include instructing the m...,11,True
5,judge must be validated against human labels,10,True
6,generate an answer from them is part of stages,1,True
7,ingest documents is part of stages,1,True


## How reliable is extraction? Sample 10 edges and check

In [7]:
sample = list(G.edges(data=True))[:10]
ok = 0
for u, v, d in sample:
    present = u.split()[0] in chunks[d["source"]].lower() and v.split()[0] in chunks[d["source"]].lower()
    ok += present
    print("OK " if present else "BAD", f"({u}) -[{d['relation']}]-> ({v})  src={d['source']}")
print(f"\n{ok}/{len(sample)} edges have both endpoints in their source chunk")

OK  (retrieval-augmented generation) -[combines]-> (retriever and language model)  src=0
OK  (retrieval-augmented generation) -[uses]-> (external knowledge base)  src=0
OK  (retrieval-augmented generation) -[passes]-> (retrieved passages to the model as context)  src=0
OK  (retrieval-augmented generation pipeline) -[has]-> (four stages)  src=0
OK  (stages) -[has]-> (four stages)  src=1
OK  (ingest documents) -[is part of]-> (stages)  src=1
OK  (split them into chunks) -[is part of]-> (stages)  src=1
OK  (index the chunks as embeddings) -[is part of]-> (stages)  src=1
OK  (retrieve the closest chunks) -[is part of]-> (stages)  src=1
OK  (generate an answer from them) -[is part of]-> (stages)  src=1

10/10 edges have both endpoints in their source chunk


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Verifiable answers | Extra bookkeeping |
| Debugging extraction becomes easy | An edge may be supported by several chunks (store a list) |
| Meets audit/compliance needs | LLMs can still cite incorrectly |

## When to use it

Enterprise and regulated settings where each statement must be traceable.

## Experiments to try

- Store the supporting sentence span, not only the chunk id.
- Keep a list of sources per edge when triples repeat.
- Fail answers with invalid citations.

## Key takeaways

- Provenance on every edge.
- Cite and verify.
- Audit extraction on samples.

---
**Next:** [34. RAPTOR](../34_raptor/raptor.ipynb)